In [ ]:
# %% [0] Geometry, Mesh & Material Configuration  --  COMSOL "Goldilocks" RF cross-section
import time
import warnings
from collections import OrderedDict

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from shapely.affinity import scale as _scale
from shapely.geometry import Polygon, box
from shapely.ops import unary_union
from skfem import MeshTri
from femwell.mesh import mesh_from_OrderedDict

warnings.filterwarnings("ignore")

# ============================================================
# 1. COMSOL PARAMETER BLOCK   (names follow the .mph parameter list)
# ============================================================
f0        = 60.0e9      # Hz    mode-analysis frequency
device_l  = 2.5e-3      # m     line length used for the S21 report

auc_w     = 60.952      # um    W_S       central signal electrode width
gap       = 4.448       # um    GAP       signal-to-ground gap
au_h      = 13.336      # um    MTX       electrode thickness
aul_w     = 70.0        # um    lateral ground width  (frozen)
cap_w     = 3.340       # um    cap_w     SiO2 cap width (sieve: <= gap - 1)
cap_h     = 1.40        # um    SiO2 cap height       (frozen)

tfln      = 0.460       # um    total thin-film LN thickness
wg_h      = 0.267       # um    ETCH_DEPTH rib height
slab_h    = tfln - wg_h # um    un-etched slab left under the rib
wg_top    = 0.80        # um    rib top base
theta     = 60.0        # deg   rib sidewall angle

box_h     = 4.7         # um    buried SiO2
si_h      = 550.0       # um    silicon substrate   (Goldilocks depth)
air_h     = 550.0       # um    air cap             (= si_h)
lat_pad   = 200.0       # um    lateral padding per side (Goldilocks zone)

basetta   = wg_h / np.tan(np.deg2rad(theta))
wg_bottom = wg_top + 2.0 * basetta

# derived x/y landmarks (y = 0 is the bottom of the LN slab)
x_sig  = auc_w / 2.0            # signal spans [-x_sig, +x_sig]
x_gi   = x_sig + gap            # ground inner edge
x_go   = x_gi + aul_w           # ground outer edge
x_dom  = x_go + lat_pad         # domain half width
x_wg   = x_sig + gap / 2.0      # rib centre (mid-gap)
y_el   = slab_h + au_h          # electrode top
y_rib  = slab_h + wg_h          # rib top

# ============================================================
# 2. MATERIALS   (COMSOL material nodes, 60 GHz)
# ============================================================
C0   = 299792458.0
EPS0 = 8.8541878128e-12
MU0  = 4.0e-7 * np.pi
ETA0 = np.sqrt(MU0 / EPS0)
DB_PER_NEPER = 20.0 * np.log10(np.e)

SIGMA_AU  = 4.56e7                                   # S/m, COMSOL "Au (Gold)" node
skin_au   = np.sqrt(2.0 / (2 * np.pi * f0 * MU0 * SIGMA_AU))
Rs_au     = 1.0 / (SIGMA_AU * skin_au)               # ohm/square

# eps_rf = (eps_xx, eps_yy, eps_zz); X-cut LN -> extraordinary axis along x
MATERIALS = {
    "LN":   {"color": "#2ecc71", "eps_rf": (28.0, 44.0, 44.0), "sigma": 1e-3,
             "desc": "LiNbO3 slab + ribs"},
    "SiO2": {"color": "#00ebfc", "eps_rf": (3.9, 3.9, 3.9),    "sigma": 1e-13,
             "desc": "SiO2 box + caps"},
    "Si":   {"color": "#7f8c8d", "eps_rf": (11.7, 11.7, 11.7), "sigma": 1e-12,
             "desc": "Silicon substrate"},
    "air":  {"color": "#ffffff", "eps_rf": (1.0, 1.0, 1.0),    "sigma": 0.0,
             "desc": "Air / vacuum"},
    "Au":   {"color": "#f39c12", "eps_rf": (1.0, 1.0, 1.0),    "sigma": 0.0,
             "desc": f"Gold electrodes (PEC + IBC, Rs = {Rs_au*1e3:.2f} mOhm/sq)"},
}

REGION_TO_MAT = {
    "corner_ring": "air", "skin_ring": "air",   # overwritten per element below
    "cap_r": "SiO2",     "cap_l": "SiO2",
    "rib_r": "LN",       "rib_l": "LN",
    "slab_near": "LN",   "slab_far": "LN",
    "box_near": "SiO2",  "box_far": "SiO2",
    "si": "Si",
    "air_gap": "air",    "air_near": "air",  "air_far": "air",
    "el_sig": "Au",      "el_gr": "Au",      "el_gl": "Au",
}
METAL_REGIONS  = ("el_sig", "el_gr", "el_gl")
SIGNAL_REGION  = "el_sig"


def mirror(p):
    return _scale(p, xfact=-1.0, yfact=1.0, origin=(0, 0))


def build_polygons():
    """COMSOL geometry sequence, rebuilt with shapely."""
    def rib(xc):
        return Polygon([(xc - wg_bottom / 2, slab_h), (xc - wg_top / 2, y_rib),
                        (xc + wg_top / 2, y_rib), (xc + wg_bottom / 2, slab_h)])

    rib_r = rib(x_wg)
    rib_l = mirror(rib_r)
    cap_r = box(x_wg - cap_w / 2, slab_h, x_wg + cap_w / 2, slab_h + cap_h).difference(rib_r)
    cap_l = mirror(cap_r)

    el_sig = box(-x_sig, slab_h, x_sig, y_el)
    el_gr  = box(x_gi, slab_h, x_go, y_el)
    el_gl  = mirror(el_gr)
    solids = unary_union([el_sig, el_gr, el_gl, cap_r, cap_l, rib_r, rib_l])

    # Conductor-surface refinement, adapted from the SKIN_SEGMENTS grading of the
    # optical notebook.  The surface current is singular at the 90 degree
    # electrode corners, so the loss integral only converges if the corners and
    # the gap-facing sidewalls carry their own mesh size.
    corners = []
    for xc in (-x_sig, x_sig, -x_gi, x_gi, -x_go, x_go):
        for yc in (slab_h, y_el):
            corners.append(box(xc - CORNER_BOX, yc - CORNER_BOX,
                               xc + CORNER_BOX, yc + CORNER_BOX))
    corner_ring = unary_union(corners).difference(solids)

    walls = []
    for xa, xb in ((x_sig, x_sig + SKIN_T), (-x_sig - SKIN_T, -x_sig),
                   (x_gi - SKIN_T, x_gi), (-x_gi, -x_gi + SKIN_T)):
        walls.append(box(xa, slab_h, xb, y_el))
    for xc in (-x_sig, x_sig, -x_gi, x_gi):
        lo, hi = (xc, xc + SKIN_REACH) if xc > 0 else (xc - SKIN_REACH, xc)
        walls.append(box(lo, y_el, hi, y_el + SKIN_T))
        walls.append(box(lo, slab_h - SKIN_T, hi, slab_h))
    skin_ring = unary_union(walls).difference(unary_union([solids, corner_ring]))

    # graded air collars: a tight one over the gap, a wider one over the electrodes
    air_gap  = box(-x_gi - COLLAR_W, slab_h, x_gi + COLLAR_W,
                   slab_h + au_h + COLLAR_H).difference(
        unary_union([solids, corner_ring, skin_ring]))
    air_near = box(-x_go - 15.0, slab_h, x_go + 15.0,
                   slab_h + au_h + 15.0).difference(
        unary_union([solids, corner_ring, skin_ring, air_gap]))
    air_far  = box(-x_dom, slab_h, x_dom, slab_h + air_h).difference(
        unary_union([solids, corner_ring, skin_ring, air_gap, air_near]))

    slab_near = box(-x_go - 15.0, 0.0, x_go + 15.0, slab_h).difference(
        unary_union([corner_ring, skin_ring]))
    slab_far  = box(-x_dom, 0.0, x_dom, slab_h).difference(slab_near)
    box_near  = box(-x_go - 15.0, -box_h, x_go + 15.0, 0.0)
    box_far   = box(-x_dom, -box_h, x_dom, 0.0).difference(box_near)
    si        = box(-x_dom, -box_h - si_h, x_dom, -box_h)

    polys = OrderedDict()
    for k, v in [("corner_ring", corner_ring), ("skin_ring", skin_ring),
                 ("cap_r", cap_r), ("cap_l", cap_l), ("rib_r", rib_r), ("rib_l", rib_l),
                 ("el_sig", el_sig), ("el_gr", el_gr), ("el_gl", el_gl),
                 ("air_gap", air_gap), ("slab_near", slab_near), ("air_near", air_near),
                 ("box_near", box_near), ("slab_far", slab_far), ("box_far", box_far),
                 ("si", si), ("air_far", air_far)]:
        polys[k] = v
    return polys


# ============================================================
# 3. MESH CONTROL
# ============================================================
# MESH_FACTOR < 1 refines everything; the reported FOMs should be checked
# against MESH_FACTOR = 0.7 before a number is trusted.
MESH_FACTOR = 1.0
COLLAR_W    = 4.0      # um, lateral reach of the fine air collar past the gap
COLLAR_H    = 4.0      # um, height of the fine air collar above the electrodes
CORNER_BOX  = 0.25     # um, half-size of the refinement box on each electrode corner
CORNER_RES  = 0.050    # um, mesh size inside those boxes  (drives the loss integral)
SKIN_T      = 0.10     # um, thickness of the conductor-surface collar
SKIN_REACH  = 5.0      # um, how far that collar runs along the top / bottom faces
SKIN_RES    = 0.100    # um, mesh size inside the collar

BASE_RES = {            # um
    "corner_ring": CORNER_RES, "skin_ring": SKIN_RES,
    "cap_r": 0.20, "cap_l": 0.20, "rib_r": 0.12, "rib_l": 0.12,
    "air_gap": 0.30, "slab_near": 0.25, "box_near": 2.0,
    "el_sig": 0.80, "el_gr": 0.80, "el_gl": 0.80,
    "air_near": 2.0, "slab_far": 6.0, "box_far": 6.0,
    "si": 30.0, "air_far": 30.0,
}
BASE_DIST = {
    "corner_ring": 0.6, "skin_ring": 1.0,
    "cap_r": 1.0, "cap_l": 1.0, "rib_r": 1.0, "rib_l": 1.0,
    "air_gap": 3.0, "slab_near": 3.0, "box_near": 5.0,
    "el_sig": 3.0, "el_gr": 3.0, "el_gl": 3.0,
    "air_near": 10.0, "slab_far": 10.0, "box_far": 10.0,
    "si": 30.0, "air_far": 30.0,
}

polygons    = build_polygons()
resolutions = {k: {"resolution": BASE_RES[k] * (1.0 if k in ("corner_ring", "skin_ring")
                                                else MESH_FACTOR),
                   "distance": BASE_DIST[k]}
               for k in polygons}

t_mesh = time.time()
raw_mesh = mesh_from_OrderedDict(polygons, resolutions=resolutions,
                                 default_resolution_min=0.4 * min(CORNER_RES, SKIN_RES),
                                 default_resolution_max=35.0)
mesh = MeshTri(raw_mesh.points[:, :2].T, raw_mesh.cells_dict["triangle"].T)
t_mesh = time.time() - t_mesh

tri_subdomains = raw_mesh.cell_data_dict["gmsh:physical"]["triangle"]
name_to_id = {n: (d[0] if hasattr(d, "__getitem__") else d)
              for n, d in raw_mesh.field_data.items()}

region_of = np.empty(mesh.nelements, dtype=object)
for raw_name, s_id in name_to_id.items():
    base_name = raw_name.split("___")[0]
    if base_name in polygons:
        region_of[tri_subdomains == s_id] = base_name
assert not any(r is None for r in region_of), "untagged elements in the mesh"
mat_of_el = np.array([REGION_TO_MAT[r] for r in region_of], dtype=object)

# The refinement collars are a meshing device, not a material: give every collar
# element the material of the layer its centroid actually sits in.
_collar = np.isin(region_of.astype(str), ("corner_ring", "skin_ring"))
if _collar.any():
    _yc = mesh.p[1, mesh.t].mean(axis=0)
    _layer = np.where(_yc < 0.0, "SiO2", np.where(_yc < slab_h, "LN", "air"))
    mat_of_el[_collar] = _layer[_collar]

is_metal  = np.isin(region_of.astype(str), METAL_REGIONS)
is_signal = region_of.astype(str) == SIGNAL_REGION

# conductor contours = facets with metal on exactly one side (the COMSOL IBC edges)
f2t          = mesh.f2t
has_two      = f2t[1] >= 0
on_contour   = has_two & (is_metal[f2t[0]] ^ is_metal[f2t[1]])
metal_side   = np.where(is_metal[f2t[0]], f2t[0], f2t[1])
contour_sig  = on_contour & is_signal[metal_side]        # COMSOL intop5
contour_gnd  = on_contour & ~is_signal[metal_side]       # COMSOL intop6
contour_all  = on_contour

print("=" * 88)
print(f"{'MATERIAL':<34} | {'eps_rf (xx, yy, zz)':<24} | {'sigma [S/m]':<12}")
print("-" * 88)
for key, d in MATERIALS.items():
    e = d["eps_rf"]
    print(f"{d['desc']:<34} | {f'({e[0]:.1f}, {e[1]:.1f}, {e[2]:.1f})':<24} | {d['sigma']:<12.3g}")
print("=" * 88)
print(f"f0 = {f0/1e9:.1f} GHz   lambda0 = {C0/f0*1e3:.3f} mm   "
      f"gold skin depth = {skin_au*1e9:.1f} nm   Rs = {Rs_au*1e3:.2f} mOhm/sq")
print(f"domain  {2*x_dom:.1f} x {si_h+box_h+slab_h+air_h:.1f} um   "
      f"(lateral padding {lat_pad:.0f} um, Si {si_h:.0f} um)")
print(f"mesh    {mesh.nelements} triangles, {mesh.nvertices} nodes, built in {t_mesh:.1f} s")
print(f"contour facets: signal {contour_sig.sum()}, ground {contour_gnd.sum()}")


def contour_groups(mask):
    """Split a set of conductor facets so every quadrature point is evaluated from
    the *dielectric* element, and return the normal pointing out of the metal."""
    out = []
    for side in (0, 1):
        other = 1 - side
        fac = np.where(mask & is_metal[f2t[other]] & ~is_metal[f2t[side]])[0]
        if fac.size == 0:
            continue
        p = mesh.p[:, mesh.facets[:, fac]]
        tan = p[:, 1, :] - p[:, 0, :]
        nrm = np.vstack([tan[1], -tan[0]])
        nrm /= np.linalg.norm(nrm, axis=0)
        midpoint = p.mean(axis=1)
        centroid = mesh.p[:, mesh.t[:, f2t[other][fac]]].mean(axis=1)
        nrm *= np.sign(((midpoint - centroid) * nrm).sum(axis=0))
        out.append((fac, side, nrm))
    return out


CONTOUR = {"sig": contour_groups(contour_sig),
           "gnd": contour_groups(contour_gnd),
           "all": contour_groups(contour_all)}

# ---- cross-section preview -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
seen, patches = set(), []
for ax, (xl, yl, ttl) in zip(axes, [
        ((-x_gi - 8, x_gi + 8), (-2.0, au_h + 3.0), "Gap region"),
        ((-x_dom, x_dom), (-box_h - 40, 60), "Full Goldilocks domain")]):
    for region in polygons:
        idx = np.where(region_of.astype(str) == region)[0]
        if idx.size == 0:
            continue
        col = MATERIALS[REGION_TO_MAT[region]]["color"]
        sub = MeshTri(mesh.p, mesh.t[:, idx])
        sub.plot(np.zeros(sub.nelements), ax=ax, shading="flat",
                 cmap=plt.matplotlib.colors.ListedColormap([col]))
        if ttl.startswith("Gap"):
            sub.draw(ax=ax, color="black", lw=0.12)
        key = REGION_TO_MAT[region]
        if key not in seen:
            patches.append(mpatches.Patch(color=col, label=f"{key}: {MATERIALS[key]['desc']}"))
            seen.add(key)
    ax.set_xlim(xl); ax.set_ylim(yl); ax.set_aspect("equal")
    ax.set_xlabel(r"$x$ ($\mu$m)"); ax.set_ylabel(r"$y$ ($\mu$m)"); ax.set_title(ttl)
axes[0].legend(handles=patches, loc="upper center", fontsize=8, framealpha=0.9)
plt.tight_layout(); plt.show()

In [ ]:
# %% [1] Anisotropic Quasi-Static Extraction (independent n_m, Z0 and the eigensolver shift)
from skfem import (Basis, BilinearForm, ElementTriP0, ElementTriP1, Functional,
                   InteriorFacetBasis, asm, condense, solve)
from skfem.helpers import grad

basis_qs = Basis(mesh, ElementTriP1())
basis_p0 = Basis(mesh, ElementTriP0())

eps_x = np.array([MATERIALS[m]["eps_rf"][0] for m in mat_of_el], dtype=float)
eps_y = np.array([MATERIALS[m]["eps_rf"][1] for m in mat_of_el], dtype=float)
eps_z = np.array([MATERIALS[m]["eps_rf"][2] for m in mat_of_el], dtype=float)
sigma = np.array([MATERIALS[m]["sigma"] for m in mat_of_el], dtype=float)
ones  = np.ones(mesh.nelements)


@BilinearForm
def laplace_aniso(u, v, w):
    return w.ex * grad(u)[0] * grad(v)[0] + w.ey * grad(u)[1] * grad(v)[1]


# every DOF inside the gold is pinned: the electrodes are equipotential bodies
dofs_sig = np.unique(basis_qs.element_dofs[:, np.where(is_signal)[0]])
dofs_gnd = np.unique(basis_qs.element_dofs[:, np.where(is_metal & ~is_signal)[0]])
dofs_D   = np.unique(np.concatenate([dofs_sig, dofs_gnd]))


def solve_static(ex, ey):
    """Laplace with V = 1 V on the signal, 0 V on both grounds."""
    K = asm(laplace_aniso, basis_qs,
            ex=basis_p0.interpolate(ex), ey=basis_p0.interpolate(ey))
    u_D = np.zeros(basis_qs.N)
    u_D[dofs_sig] = 1.0
    Kc, fc, uc, I = condense(K, np.zeros(basis_qs.N), x=u_D, D=dofs_D)
    u = uc.copy()
    u[I] = solve(Kc, fc)
    # variational charge recovery: Q/eps0 = sum of the residual over the pinned DOFs.
    # This is exact for the discrete solution, unlike integrating dV/dn over a
    # contour that runs through the re-entrant corners of the electrodes.
    return u, EPS0 * float((K @ u)[dofs_sig].sum())


t_qs = time.time()
V_eps, C_eps = solve_static(eps_x, eps_y)      # F/m, the um -> m factors cancel
V_air, C_air = solve_static(ones, ones)

n_m_qs = np.sqrt(C_eps / C_air)
Z0_qs  = 1.0 / (C0 * np.sqrt(C_eps * C_air))
L_qs   = 1.0 / (C0 ** 2 * C_air)


# ---- Wheeler perturbation loss from the air-problem surface charge ----------
# For a quasi-TEM line  Jsz(l)/I = rho_air(l)/Q_air, so  R' = Rs * contour(|Jsz/I|^2).
@Functional
def dVdn_squared(w):
    return (w.gu.grad[0] * w.nx + w.gu.grad[1] * w.ny) ** 2


def contour_static(key, field):
    total = 0.0
    for fac, side, nrm in CONTOUR[key]:
        fb = InteriorFacetBasis(mesh, ElementTriP1(), facets=fac, side=side)
        total += dVdn_squared.assemble(fb, gu=fb.interpolate(field),
                                       nx=nrm[0][:, None], ny=nrm[1][:, None])
    return total


S_charge   = C_air / EPS0                          # = contour integral of dV/dn on the signal
R_prime_qs = Rs_au * 1e6 * contour_static("all", V_air) / S_charge ** 2
alpha_qs   = R_prime_qs / (2.0 * Z0_qs)            # Np/m
alpha_qs_db = DB_PER_NEPER * alpha_qs / 100.0
t_qs = time.time() - t_qs

print("=" * 68)
print("  QUASI-STATIC EXTRACTION  (Laplace with the anisotropic eps_RF tensor)")
print("=" * 68)
print(f"  C                 {C_eps*1e12:12.4f} pF/m")
print(f"  C_air             {C_air*1e12:12.4f} pF/m")
print(f"  L                 {L_qs*1e9:12.4f} nH/m")
print(f"  n_m               {n_m_qs:12.5f}")
print(f"  Z0                {Z0_qs:12.4f} ohm")
print(f"  R' (perturbation) {R_prime_qs:12.2f} ohm/m")
print(f"  alpha_c (perturb) {alpha_qs_db:12.4f} dB/cm")
print(f"  solved in         {t_qs:12.2f} s")
print("=" * 68)

n_guess = n_m_qs      # eigensolver shift; COMSOL used a flat 1.95 for every geometry

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
mesh.plot(V_eps, ax=axes[0], shading="gouraud", cmap="coolwarm", colorbar=True)
axes[0].set_title(r"$V(x,y)$ with $\epsilon_{RF}$ = diag(28, 44)")
mesh.plot(V_air, ax=axes[1], shading="gouraud", cmap="coolwarm", colorbar=True)
axes[1].set_title(r"$V_{air}(x,y)$  ($\epsilon$ = 1, sets $L$ and $J_{sz}$)")
for ax in axes:
    ax.set_xlim([-x_gi - 6, x_gi + 6]); ax.set_ylim([-2.0, au_h + 3.0])
    ax.set_aspect("equal"); ax.set_xlabel(r"$x$ ($\mu$m)"); ax.set_ylabel(r"$y$ ($\mu$m)")
plt.tight_layout(); plt.show()

In [ ]:
# %% [2] RF Vector Eigensolver (anisotropic, PEC electrodes = COMSOL IBC edges)
import scipy.constants
from skfem import BilinearForm, ElementTriN1, ElementTriP0, ElementTriP1, condense, solve
from skfem.helpers import curl, dot, grad
from skfem.utils import solver_eigen_scipy

NUM_MODES = 20           # COMSOL asked ARPACK for 40; the filters below need far fewer
SHIFT     = n_guess      # "search for modes around shift"; COMSOL used a flat 1.95

wl_rf = C0 / f0 * 1e6                 # um
k0    = 2.0 * np.pi / wl_rf           # 1/um
omega = 2.0 * np.pi * f0
UM    = 1e-6

element_rf = ElementTriN1() * ElementTriP1()
basis_rf   = basis_p0.with_element(element_rf)
basis_eps  = basis_rf.with_element(ElementTriP0())


@BilinearForm(dtype=complex)
def aform(e_t, e_z, v_t, v_z, w):
    return (curl(e_t) * curl(v_t) / k0 ** 2
            - (w.exx * e_t[0] * v_t[0] + w.eyy * e_t[1] * v_t[1])
            + dot(grad(e_z), v_t)
            + (w.exx * e_t[0] * grad(v_z)[0] + w.eyy * e_t[1] * grad(v_z)[1])
            - w.ezz * e_z * v_z * k0 ** 2)


@BilinearForm(dtype=complex)
def bform(e_t, e_z, v_t, v_z, w):
    return -dot(e_t, v_t) / k0 ** 2


t_asm = time.time()
A = aform.assemble(basis_rf, exx=basis_eps.interpolate(eps_x),
                   eyy=basis_eps.interpolate(eps_y), ezz=basis_eps.interpolate(eps_z))
B = bform.assemble(basis_rf)
t_asm = time.time() - t_asm

# PEC everywhere the field must vanish: inside + on the gold, and on the outer window
dofs_metal = np.unique(basis_rf.element_dofs[:, np.where(is_metal)[0]])
dofs_outer = basis_rf.get_dofs(facets=mesh.boundary_facets()).flatten()
dofs_pec   = np.unique(np.concatenate([dofs_metal, dofs_outer]))

t_eig = time.time()
lams, xs = solve(*condense(-A, -B, D=dofs_pec, x=basis_rf.zeros(dtype=complex)),
                 solver=solver_eigen_scipy(k=NUM_MODES, sigma=k0 ** 2 * SHIFT ** 2))
t_eig = time.time() - t_eig

# femwell scaling: the solved z-unknown is (j*beta/k0^2) * E_z
idx_t, idx_z = basis_rf.split_indices()
xs[idx_z, :] /= 1j * np.sqrt(lams[np.newaxis, :] / k0 ** 4)

betas = np.sqrt(lams.astype(complex))          # 1/um
neffs = betas / k0

print(f"assembled {A.shape[0]} DOF in {t_asm:.1f} s   |   "
      f"{NUM_MODES} modes around n = {SHIFT:.4f} in {t_eig:.1f} s")

In [ ]:
# %% [3] Mode Diagnostics and the COMSOL Three-Filter Pipeline
from skfem import Functional, InteriorFacetBasis

# the residual of the Gauss row over the pinned gold DOFs is the conductor charge;
# charge conservation on the contour then gives the longitudinal current exactly:
#   j*beta*Isig + d(Jst)/dl + j*omega*Qsig = 0   ->   Isig = (omega/beta) * Qsig
xs_raw = xs.copy()
xs_raw[idx_z, :] *= 1j * np.sqrt(lams[np.newaxis, :] / k0 ** 4)

_is_z = np.zeros(basis_rf.N, dtype=bool); _is_z[idx_z] = True
dofs_sig_z = np.unique(basis_rf.element_dofs[:, np.where(is_signal)[0]])
dofs_sig_z = dofs_sig_z[_is_z[dofs_sig_z]]
dofs_gnd_z = np.unique(basis_rf.element_dofs[:, np.where(is_metal & ~is_signal)[0]])
dofs_gnd_z = dofs_gnd_z[_is_z[dofs_gnd_z]]


@Functional(dtype=complex)
def f_poynting(w):                 # COMSOL emw.Poavz over every non-metal domain
    Hx = (w.ez.grad[1] / UM - 1j * w.beta * w.et[1]) / (-1j * omega * MU0)
    Hy = (1j * w.beta * w.et[0] - w.ez.grad[0] / UM) / (-1j * omega * MU0)
    return 0.5 * np.real(w.et[0] * np.conj(Hy) - w.et[1] * np.conj(Hx)) * UM ** 2


@Functional(dtype=complex)
def f_dielectric(w):
    return 0.5 * w.sig * (np.abs(w.et[0]) ** 2 + np.abs(w.et[1]) ** 2
                          + np.abs(w.ez) ** 2) * UM ** 2


@Functional(dtype=complex)
def f_Js2(w):                      # |Js|^2 = |Jsz|^2 + |Hz|^2 , the IBC loss integrand
    dEz_dn = (w.ez.grad[0] * w.nx + w.ez.grad[1] * w.ny) / UM
    E_n = w.et[0] * w.nx + w.et[1] * w.ny
    Jz = (-dEz_dn + 1j * w.beta * E_n) / (-1j * omega * MU0)
    Hz = (w.et.curl / UM) / (-1j * omega * MU0)
    return (np.abs(Jz) ** 2 + np.abs(Hz) ** 2) * UM


@Functional(dtype=complex)
def f_Jsz(w):
    dEz_dn = (w.ez.grad[0] * w.nx + w.ez.grad[1] * w.ny) / UM
    E_n = w.et[0] * w.nx + w.et[1] * w.ny
    return (-dEz_dn + 1j * w.beta * E_n) / (-1j * omega * MU0) * UM


def contour_mode(form, key, x, beta_m):
    total = 0.0 + 0j
    for fac, side, nrm in CONTOUR[key]:
        fb = InteriorFacetBasis(mesh, element_rf, facets=fac, side=side)
        fields = fb.interpolate(x)
        total += form.assemble(fb, et=fields[0], ez=fields[1],
                               nx=nrm[0][:, None], ny=nrm[1][:, None], beta=beta_m)
    return total


# COMSOL lineop1: integral of Ex across the gap at mid-electrode height
_xpath = np.linspace(-x_gi, -x_sig, 801)
_probe = basis_rf.split(np.zeros(basis_rf.N))[0][1].probes(
    np.vstack([_xpath, np.full_like(_xpath, slab_h + au_h / 2.0)]))


def gap_voltage(x):
    Ex = np.asarray(_probe @ x[idx_t]).reshape(2, -1)[0]
    return np.trapezoid(Ex, _xpath) * UM


t_diag = time.time()
rows = []
for i in range(NUM_MODES):
    beta_m = betas[i] * 1e6                                  # rad/m
    if abs(beta_m) < 1e-9:
        continue
    x, xr = xs[:, i], xs_raw[:, i]
    fields = basis_rf.interpolate(x)
    P = abs(float(np.real(f_poynting.assemble(basis_rf, et=fields[0], ez=fields[1],
                                              beta=beta_m))))
    if P < 1e-30:
        continue
    res = A @ xr
    Q_sig = EPS0 * abs(res[dofs_sig_z].sum()) * 1e-6
    Q_gnd = EPS0 * abs(res[dofs_gnd_z].sum()) * 1e-6
    I_sig = omega / abs(beta_m) * Q_sig
    I_gnd = omega / abs(beta_m) * Q_gnd
    V_gap = abs(gap_voltage(x))
    Z_PI = 2.0 * P / I_sig ** 2 if I_sig > 0 else np.inf
    Z_VP = V_gap ** 2 / (2.0 * P)
    score = I_sig / (I_sig + I_gnd) if (I_sig + I_gnd) > 0 else np.nan
    div = abs(Z_PI - Z_VP) / max(0.5 * (Z_PI + Z_VP), 1e-30)
    rows.append(dict(index=i, neff=complex(neffs[i]), P=P, I_sig=I_sig, I_gnd=I_gnd,
                     V_gap=V_gap, Z_PI=Z_PI, Z_VP=Z_VP, score=score, div=div,
                     beta_m=beta_m))

F1 = [r for r in rows if 0.0 < r["Z_PI"] < 150.0]
F2 = [r for r in F1 if 0.45 <= r["score"] <= 0.55]
F3 = sorted(F2, key=lambda r: r["div"])
t_diag = time.time() - t_diag

print("=" * 98)
print(f"{'i':>3} | {'Re(n_eff)':>10} | {'Z0_IBC':>12} | {'Z0_volt_IBC':>12} | "
      f"{'Mode_Score':>10} | {'divergence':>10} | {'verdict':<9}")
print("-" * 98)
for r in sorted(rows, key=lambda q: -q["neff"].real):
    verdict = ("SELECTED" if F3 and r is F3[0] else
               "pass F2" if r in F2 else "cut F2" if r in F1 else "cut F1")
    fmt = (lambda v: f"{v:12.4f}" if abs(v) < 1e5 else f"{v:12.3e}")
    print(f"{r['index']:3d} | {r['neff'].real:10.5f} | {fmt(r['Z_PI'])} | {fmt(r['Z_VP'])} | "
          f"{r['score']:10.5f} | {r['div']*100:9.3f}% | {verdict:<9}")
print("=" * 98)
print(f"filter 1  0 < Z0_IBC < 150 ohm        : {len(rows):3d} -> {len(F1):3d}")
print(f"filter 2  0.45 <= Mode_Score <= 0.55  : {len(F1):3d} -> {len(F2):3d}")
print(f"filter 3  min |Z0_IBC - Z0_voltage|   : {len(F2):3d} -> {min(len(F2),1):3d}    ({t_diag:.1f} s)")
if not F3:
    raise RuntimeError("no quasi-TEM CPW mode survived; raise NUM_MODES or check the shift")
fundamental = F3[0]

# ---- mode gallery ----------------------------------------------------------
_show = [r for r in sorted(rows, key=lambda q: -q["neff"].real)][:8]
if _show:
    _cols = 4
    _rws = int(np.ceil(len(_show) / _cols))
    fig, axes = plt.subplots(_rws, _cols, figsize=(16, 3.2 * _rws))
    axes = np.atleast_1d(axes).flatten()
    for ax, r in zip(axes, _show):
        fl = basis_rf.interpolate(xs[:, r["index"]])
        Ex = np.mean(np.abs(np.asarray(fl[0])[0]) ** 2, axis=-1)
        Ex = Ex / (Ex.max() if Ex.max() > 0 else 1.0)
        mesh.plot(np.sqrt(Ex), ax=ax, shading="flat", cmap="inferno")
        tag = ("SELECTED" if F3 and r is F3[0] else
               "pass F2" if r in F2 else "cut F2" if r in F1 else "cut F1")
        ax.set_title(f"mode {r['index']}: n = {r['neff'].real:.4f}\n"
                     f"Z0 = {r['Z_PI']:.3g} | score = {r['score']:.3f}\n{tag}", fontsize=8)
        ax.set_xlim([-x_go - 10, x_go + 10]); ax.set_ylim([-8, au_h + 10])
        ax.set_aspect("equal")
    for ax in axes[len(_show):]:
        fig.delaxes(ax)
    plt.suptitle(r"$|E_x|$ of every candidate mode (each panel normalised)", y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
# %% [4] Figures of Merit: n_m, Z0, RF Attenuation
i_sel  = fundamental["index"]
x_sel  = xs[:, i_sel]
beta_m = fundamental["beta_m"]
P_sel  = fundamental["P"]

fields = basis_rf.interpolate(x_sel)
J2     = float(np.real(contour_mode(f_Js2, "all", x_sel, beta_m)))     # contour |Js|^2 dl
P_diel = float(np.real(f_dielectric.assemble(basis_rf, et=fields[0], ez=fields[1],
                                             sig=basis_eps.interpolate(sigma))))

# ---------------------------------------------------------------------------
# First-order IBC correction.
#
# The eigensolve above uses PEC electrodes, so it returns the EXTERNAL
# inductance only.  A real conductor presents Zs = (1 + j) * Rs, and the very
# same contour integral that produces the loss resistance also produces an
# internal (surface) inductance:
#
#     R' + j*w*L_int = Zs * contour(|Js/I|^2 dl)   ->   L_int = R'/w
#
# n_m and Z0 both scale with sqrt(L_ext + L_int), so for thick electrodes over
# a narrow gap this is a several-percent shift, not a rounding error.
# ---------------------------------------------------------------------------
G_contour = J2 / fundamental["I_sig"] ** 2          # 1/m
R_prime   = Rs_au * G_contour                       # ohm/m
n_m_pec   = fundamental["neff"].real
Z0_pec    = fundamental["Z_PI"]
L_ext     = Z0_pec * n_m_pec / C0                   # H/m
L_int     = R_prime / omega                         # H/m
kappa     = np.sqrt(1.0 + L_int / L_ext)

n_m  = n_m_pec * kappa
Z0   = Z0_pec * kappa
Z_VP = fundamental["Z_VP"] * kappa
Z_VI = fundamental["V_gap"] / fundamental["I_sig"] * kappa

alpha_c  = R_prime / (2.0 * Z0)                     # Np/m, conductor (IBC) loss
alpha_d  = P_diel / (2.0 * P_sel)                   # Np/m, dielectric loss
alpha_np = alpha_c + alpha_d

alpha_c_db  = DB_PER_NEPER * alpha_c / 100.0
alpha_d_db  = DB_PER_NEPER * alpha_d / 100.0
alpha_db_cm = DB_PER_NEPER * alpha_np / 100.0
neff_im     = -alpha_np / (2 * np.pi * f0 / C0)     # COMSOL emw.neff imaginary part
S21_db      = -DB_PER_NEPER * alpha_np * device_l

# mesh-quality probe: the direct contour integral of Jsz has to reproduce the
# charge-conservation current.  A ratio away from 1 means the conductor contour
# (in particular its re-entrant corners) is still under-resolved.
I_direct = abs(contour_mode(f_Jsz, "sig", x_sel, beta_m))

print("=" * 66)
print("           2D RF BASELINE  --  unetched CPW at 60 GHz")
print("=" * 66)
print(f"  selected mode                      {i_sel:>14d}")
print(f"  emw.neff                    {n_m:>12.5f} {neff_im:+.6f}i")
print(f"  microwave index   n_m              {n_m:>14.5f}")
print(f"  Z0_IBC          (power-current)    {Z0:>14.4f} ohm")
print(f"  Z0_voltage_IBC  (voltage-power)    {Z_VP:>14.4f} ohm")
print(f"  Z0              (voltage-current)  {Z_VI:>14.4f} ohm")
print(f"  Mode_Score                         {fundamental['score']:>14.5f}")
print(f"  quasi-TEM divergence               {fundamental['div']*100:>13.3f} %")
print("-" * 66)
print(f"  conductor loss (IBC)               {alpha_c_db:>14.4f} dB/cm")
print(f"  dielectric loss                    {alpha_d_db:>14.4f} dB/cm")
print(f"  Attenuation                        {alpha_db_cm:>14.4f} dB/cm")
print(f"  S21 over {device_l*1e3:>4.1f} mm                    {S21_db:>14.4f} dB")
print("-" * 66)
print(f"  R'                                 {R_prime:>14.2f} ohm/m")
print(f"  L_ext                              {L_ext*1e9:>14.3f} nH/m")
print(f"  L_int (IBC surface inductance)     {L_int*1e9:>14.3f} nH/m")
print(f"  kappa = sqrt(1 + L_int/L_ext)      {kappa:>14.4f}")
print("-" * 66)
print(f"{'':<28}{'PEC solve':>12}{'quasi-static':>14}{'IBC corr.':>12}")
print(f"{'  n_m':<28}{n_m_pec:>12.5f}{n_m_qs:>14.5f}{n_m:>12.5f}")
print(f"{'  Z0 [ohm]':<28}{Z0_pec:>12.4f}{Z0_qs:>14.4f}{Z0:>12.4f}")
print(f"{'  alpha_c [dB/cm]':<28}{DB_PER_NEPER*R_prime/(2*Z0_pec)/100:>12.4f}"
      f"{alpha_qs_db:>14.4f}{alpha_c_db:>12.4f}")
print(f"{'  |Isig| contour / exact':<28}{I_direct/fundamental['I_sig']:>12.4f}")
print("=" * 66)

# ---- selected mode and its surface current ---------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

_fl = basis_rf.interpolate(x_sel)
_Ex = np.mean(np.abs(np.asarray(_fl[0])[0]) ** 2, axis=-1)
_Ex = np.sqrt(_Ex / (_Ex.max() if _Ex.max() > 0 else 1.0))
mesh.plot(_Ex, ax=axes[0], shading="flat", cmap="inferno", colorbar=True)
axes[0].set_xlim([-x_gi - 10, x_gi + 10]); axes[0].set_ylim([-3, au_h + 4])
axes[0].set_aspect("equal")
axes[0].set_title(rf"selected mode $|E_x|$, $n_m$ = {n_m:.4f}")

_px, _py, _pj = [], [], []
for _fac, _side, _nrm in CONTOUR["all"]:
    _fb = InteriorFacetBasis(mesh, element_rf, facets=_fac, side=_side)
    _f = _fb.interpolate(x_sel)
    _dn = (np.asarray(_f[1].grad[0]) * _nrm[0][:, None]
           + np.asarray(_f[1].grad[1]) * _nrm[1][:, None]) / UM
    _en = (np.asarray(_f[0])[0] * _nrm[0][:, None]
           + np.asarray(_f[0])[1] * _nrm[1][:, None])
    _jz = (-_dn + 1j * beta_m * _en) / (-1j * omega * MU0)
    _mid = mesh.p[:, mesh.facets[:, _fac]].mean(axis=1)
    _px.append(_mid[0]); _py.append(_mid[1])
    _pj.append(np.abs(_jz).mean(axis=1) / fundamental["I_sig"])
_px, _py, _pj = np.concatenate(_px), np.concatenate(_py), np.concatenate(_pj)
_sc = axes[1].scatter(_px, _py, c=np.log10(np.maximum(_pj, 1e-12)), s=6, cmap="viridis")
plt.colorbar(_sc, ax=axes[1], label=r"$\log_{10}|J_{sz}/I|$  (1/m)")
axes[1].set_xlim([-x_gi - 10, x_gi + 10]); axes[1].set_ylim([-3, au_h + 4])
axes[1].set_aspect("equal")
axes[1].set_title("surface current on the IBC contour\n(the corners drive $R'$)")
for ax in axes:
    ax.set_xlabel(r"$x$ ($\mu$m)"); ax.set_ylabel(r"$y$ ($\mu$m)")
plt.tight_layout(); plt.show()

In [ ]:
# %% [5] Cross-check against the recorded COMSOL sweep
# Rows copied verbatim from the 500-LHS COMSOL RF output.  Edit the parameter
# block in cell [0] to one of these geometries and re-run cells [0]-[4]; this
# cell then reports the agreement for whichever row matches.
#   columns: auc_w  gap  au_h  cap_w  wg_h | Re(neff)  Z0_IBC  Z0_voltage_IBC  Mode_Score  Attenuation
COMSOL_ROWS = [
    (60.952, 4.448, 13.336, 3.340, 0.267, 1.6751, 21.325, 21.393, 0.50223, 5.6598),
    (78.739, 5.662, 13.680, 3.145, 0.319, 1.7169, 23.099, 23.203, 0.50282, 5.7375),
    (63.592, 5.053, 10.779, 3.152, 0.283, 1.7436, 24.379, 24.465, 0.50259, 5.4279),
    (75.167, 6.486, 12.843, 2.751, 0.179, 1.7876, 24.756, 24.876, 0.50314, 5.5161),
    (78.024, 7.285, 14.935, 5.671, 0.280, 1.7634, 24.927, 25.056, 0.50316, 5.4148),
]

here = np.array([auc_w, gap, au_h, cap_w, wg_h])
match = None
for row in COMSOL_ROWS:
    if np.allclose(here, np.array(row[:5]), rtol=2e-3, atol=2e-3):
        match = row
        break

print("=" * 74)
print("  COMSOL CROSS-CHECK")
print("=" * 74)
if match is None:
    print("  the geometry in cell [0] is not one of the recorded rows; set it to")
    print(f"  one of these (auc_w, gap, au_h, cap_w, wg_h) and re-run cells [0]-[4]:")
    for row in COMSOL_ROWS:
        print(f"      {row[0]:7.3f} {row[1]:6.3f} {row[2]:7.3f} {row[3]:6.3f} {row[4]:6.3f}"
              f"   ->  n_m {row[5]:.4f}, Z0 {row[6]:.3f} ohm, alpha {row[9]:.4f} dB/cm")
else:
    _, _, _, _, _, n_ref, z_ref, zv_ref, sc_ref, a_ref = match
    print(f"{'quantity':<28}{'this code':>12}{'COMSOL':>12}{'delta':>12}")
    print("-" * 74)
    for label, mine, ref in [
            ("n_m",                     n_m,   n_ref),
            ("Z0_IBC [ohm]",            Z0,    z_ref),
            ("Z0_voltage_IBC [ohm]",    Z_VP,  zv_ref),
            ("Mode_Score",              fundamental["score"], sc_ref),
            ("Attenuation [dB/cm]",     alpha_db_cm, a_ref)]:
        print(f"{'  '+label:<28}{mine:>12.5f}{ref:>12.5f}{(mine/ref-1)*100:>11.2f}%")
    print("-" * 74)
    print(f"{'  L (= Z0*n_m/c) [nH/m]':<28}{Z0*n_m/C0*1e9:>12.4f}"
          f"{z_ref*n_ref/C0*1e9:>12.4f}{(Z0*n_m)/(z_ref*n_ref)*100-100:>11.2f}%")
    print(f"{'  C (= n_m/(Z0*c)) [pF/m]':<28}{n_m/(Z0*C0)*1e12:>12.4f}"
          f"{n_ref/(z_ref*C0)*1e12:>12.4f}{(n_m*z_ref)/(Z0*n_ref)*100-100:>11.2f}%")
print("=" * 74)